In [1]:
import re
import numpy as np
import pandas as pd
import os
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../notebooks")
print(os.listdir(DATA_DIR))
members = pd.read_csv(
    DATA_DIR / "members.csv",
    parse_dates=["join_date"]
)

renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    parse_dates=["expiry_date", "renewal_deadline"]
)

events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    parse_dates=["event_time"]
)

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

['weather.csv', 'weather3.csv', 'DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb', 'members.csv', 'renewal_opportunities.csv', 'member_events.csv', 'weather2.csv']


1
Using members, group by acquisition_channel.

Write a function that receives the annual_fee Series and returns the difference between its highest and lowest values. Produce one row per channel with an output column named fee_range.

In [2]:
members.head()

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


In [3]:
def get_fee_range(annual_fees):
    return annual_fees.max() - annual_fees.min()

curr_members = members.copy()
fee_range = (
    curr_members.groupby('acquisition_channel')['annual_fee']
    .apply(get_fee_range)
    .reset_index(name="fee_range")
)
fee_range

,acquisition_channel,fee_range
0,APP,1000
1,STORE,1000
2,WEB,1000


2.
Using members, group by city.

The function should receive the group DataFrame. Among members who were "PAID" on 9 October 2025, calculate the percentage who are "EXPIRED" on 9 October 2026.

Return one scalar per city, named expired_previous_paid_pct. If a city has no previously paid members, return NaN. Members previously marked "NOT_JOINED" or "EXPIRED" must not enter the denominator.

In [4]:
def get_expired_percentage(city_members):
    previously_paid = city_members['status_2025_10_09'] == "PAID"
    denominator = previously_paid.sum()

    if denominator == 0:
        return np.nan

    expired_now = city_members['status_2026_10_09'] == "EXPIRED"
    return (expired_now & previously_paid).sum() / denominator * 100

In [5]:
curr_members = members.copy()
expired_previous_paid_pct = (
    curr_members.groupby('city')[['status_2025_10_09', 'status_2026_10_09']]
    .apply(get_expired_percentage)
    .reset_index(name="expired_previous_paid_pct")
)
expired_previous_paid_pct

,city,expired_previous_paid_pct
0,Ahmedabad,23.344371
1,Bengaluru,22.067901
2,Delhi,25.000000
3,Gurugram,24.267101
4,Hyderabad,24.344023
5,Jaipur,25.077399
6,Mumbai,24.644550
7,Pune,29.093800


3
Using members, group by acquisition_channel.

Write a function that receives the group DataFrame and returns the mean annual_fee of members currently marked "PAID". Produce current_paid_average_fee, with one row per acquisition channel.

In [9]:
def get_current_paid_average_fee(channel_members):
    currently_paid = channel_members.loc[
        channel_members['status_2026_10_09'] == "PAID", 'annual_fee'
    ]

    if currently_paid.empty:
        return np.nan

    return currently_paid.mean()

curr_members = members.copy()
current_paid_average_fee = (
    curr_members.groupby('acquisition_channel')[['status_2026_10_09', 'annual_fee']]
    .apply(get_current_paid_average_fee)
    .reset_index(name="current_paid_average_fee")
)
current_paid_average_fee

,acquisition_channel,current_paid_average_fee
0,APP,1803.505097
1,STORE,1792.863440
2,WEB,1789.799562


4
Using members, group by city.

Write a function that returns a named summary Series containing:
- registered_members: number of rows in the group.
- currently_paid: number currently marked "PAID".
- current_paid_pct: currently paid members as a percentage of all registered members in that city.

The final output should have one row per city and these three columns. This percentage is a billing-snapshot measure, not a renewal rate.

In [9]:
def summarize_city_members(city_members):
    currently_paid = city_members['status_2026_10_09'] == "PAID"
    registered_members = len(city_members)
    paid_count = currently_paid.sum()

    
    city_members["registered_members"]= registered_members
    city_members["currently_paid"]= paid_count
    city_members["current_paid_pct"]= round(paid_count / registered_members * 100,2)
    

city_paid_summary = (
    members.groupby('city')[['status_2026_10_09']]
    .apply(summarize_city_members)
    .reset_index()
)
city_paid_summary

,city,registered_members,currently_paid,current_paid_pct
0,Ahmedabad,1021.0,725.0,71.01
1,Bengaluru,1064.0,771.0,72.46
2,Delhi,1071.0,754.0,70.40
3,Gurugram,1055.0,755.0,71.56
4,Hyderabad,1124.0,789.0,70.20
5,Jaipur,1129.0,778.0,68.91
6,Mumbai,1076.0,754.0,70.07
7,Pune,1040.0,698.0,67.12


5
Using members, group by city and pass the annual_fee Series to your function.

Return a Series showing each member's annual fee as a percentage of the sum of listed annual fees in that city. Assign it back to members as city_fee_share_pct.

Preserve every original row index. This is a share of listed fees, not a share of collected revenue. If a total is zero, return zero for every row in that group.

In [11]:
def get_city_fee_share(annual_fees):
    total_fees = annual_fees.sum()
    if total_fees == 0:
        return annual_fees * 0

    return annual_fees / total_fees * 100

curr_members = members.copy()
city_fee_share = (
    members.groupby('city', group_keys=False)['annual_fee']
    .apply(get_city_fee_share)
)
members.loc[city_fee_share.index, 'city_fee_share_pct'] = city_fee_share
curr_members

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_fee_share_pct
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,0.079970
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,0.128952
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,0.077350
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,0.078587
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,0.072860
...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,0.073305
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,0.077350
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,0.080291
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,0.133319


6. Using members, group by city and pass the join_date Series to your function.

Return the number of whole days between each member's joining date and the earliest joining date in that city. Assign the returned Series as days_after_city_first_join.

The earliest joiner in each city should receive zero. Preserve the original indexes.

In [18]:
members.groupby('city')[['join_date','city']].transform('min')

,join_date,city
0,2023-09-01,Gurugram
1,2023-09-01,Bengaluru
2,2023-09-01,Bengaluru
3,2023-09-01,Delhi
4,2023-09-01,Jaipur
...,...,...
8575,2023-09-01,Hyderabad
8576,2023-09-01,Bengaluru
8577,2023-09-01,Pune
8578,2023-09-01,Gurugram


In [16]:
curr_members = members.copy()
def get_whole_days(join_dates):
    # print(join_dates)
    return (join_dates - min(join_dates)).dt.days


curr_members["days_after_city_first_join"] = curr_members.groupby('city')['join_date'].transform(get_whole_days)
print(curr_members.shape)
display(curr_members)

(8580, 10)


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,days_after_city_first_join
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,21
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,734
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,12
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,764
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,25
...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,891
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,5
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,922
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,7


7. Using members, group by city.

For each city, calculate the mean annual fee among currently paid members. Your function should return a Series with one value for every original member row:

For a currently paid member, return their fee minus that city's mean fee among currently paid members.
For every other member, return NaN.
Assign the result as paid_fee_difference. Do not remove unpaid rows from the returned Series.

In [7]:
curr_members = members.copy()

def avg_annual_fee(curr_members):
    is_paid = curr_members['status_2026_10_09'] == "PAID"
    paid_members_mean = curr_members.loc[is_paid, 'annual_fee'].mean()
    res = curr_members['annual_fee']- paid_members_mean
    # print(res)

    return res.where(is_paid,np.nan)

curr_members["paid_fee_difference"] = (
    curr_members
    .groupby('city', group_keys=False)[['status_2026_10_09', 'annual_fee']]
    .apply(avg_annual_fee)
)
# curr_members["paid_fee_difference"]
curr_members

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,paid_fee_difference
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,-283.443709
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,677.042802
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,-322.957198
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,-287.798408
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,-320.051414
...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,-301.647655
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,-322.957198
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,-295.128940
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,716.556291


8. Using members, group by city and send the group DataFrame to your function.

Return a DataFrame with the same row indexes and two columns:

city_paid_member_count: the number of currently paid members in the city, repeated on every member row.
paid_fee_share_pct: each currently paid member's share of that city's total listed fees among currently paid members. Use zero for members not currently paid and for a group with no paid fee total.
Assign both returned columns back to the original rows. Explain why summing city_paid_member_count over those rows would overcount members.

In [11]:
curr_members = members.copy()

def payment_details(df):
    is_paid = df['status_2026_10_09'] == "PAID"
    # paid_members = curr_members.loc[is_paid, 'annual_fee']
    # paid_members_count = curr_members.loc[is_paid].count()
    paid_members_count = is_paid.sum()
    paid_members_sum = df.loc[is_paid, 'annual_fee'].sum()
    if paid_members_sum:
        fee_share = (df['annual_fee'] / paid_members_sum * 100).where(is_paid, 0)
    else:
        fee_share = pd.Series(0,index=df.index)

    return pd.DataFrame({"city_paid_member_count": paid_members_count,
                         "paid_fee_share_pct": fee_share})



res = curr_members.groupby('city', group_keys=False)[['status_2026_10_09', 'annual_fee']].apply(payment_details)
print(res)
curr_members[["city_paid_member_count","paid_fee_share_pct"]] = res[["city_paid_member_count","paid_fee_share_pct"]]

curr_members


      city_paid_member_count  paid_fee_share_pct
0                        755            0.111388
1                        771            0.177899
2                        771            0.106711
3                        754            0.111264
4                        778            0.105920
...                      ...                 ...
8575                     789            0.105511
8576                     771            0.106711
8577                     698            0.119700
8578                     755            0.185696
8579                     771            0.177899

[8580 rows x 2 columns]


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_paid_member_count,paid_fee_share_pct
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,755,0.111388
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,771,0.177899
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,771,0.106711
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,754,0.111264
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,778,0.105920
...,...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,789,0.105511
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,771,0.106711
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,698,0.119700
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,755,0.185696


9.
Using renewals, group by renewal_number.

Write a function receiving the offered_discount Series. Return the percentage of records with a discount greater than zero, rounded to two decimal places.

Produce one row per renewal number, with a column named discount_offer_pct.

In [12]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


In [19]:
curr_renewals = renewals.copy()
def percentage_records(ds):
    is_discount = ds > 0
    total_records = ds.count()
    discounted_users_count = is_discount.sum()
    return round(discounted_users_count/total_records * 100,2)



# curr_renewals['discount_offer_pct'] = renewals.groupby("renewal_number")["offered_discount"].apply(percentage_records)
result = (curr_renewals.groupby("renewal_number")["offered_discount"].apply(percentage_records)
 .reset_index(name="discount_offer_pct"))

display(result)

,renewal_number,discount_offer_pct
0,1,7.88
1,2,10.82
2,3,11.72


10.
Using renewals, group by renewal_number and pass the group DataFrame.

Among records where offered_discount > 0, calculate the mean of renewal_fee - offered_discount.

Return one scalar per group, named average_discounted_offer. This is an average offered price, not average cash received.

In [24]:
curr_renewals = renewals.copy()

def get_average_discounted_offer(df):
    is_discount = df["offered_discount"] > 0
    discounted_renewal_fee_mean = (df.loc[is_discount, 'renewal_fee']-
                                   df.loc[is_discount, 'offered_discount']).mean()
    return discounted_renewal_fee_mean

curr_renewals["average_discounted_offer"] = curr_renewals.groupby("renewal_number")[["offered_discount","renewal_fee"]].apply(get_average_discounted_offer)

curr_renewals


,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,average_discounted_offer
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,NaN
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,1653.174950
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,1596.010870
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,1617.720379
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,NaN
...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,NaN
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,NaN
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,NaN
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,NaN
